In [ ]:
def synthetic_stable_layer(z, L_ob=-50.0, u_star=0.3, theta_star=0.1, z0=0.05):
    """Generate synthetic stable boundary layer profile (MOST-based).
    
    Parameters
    ----------
    z : array
        Height grid (m)
    L_ob : float
        Obukhov length (m), negative for stable
    u_star : float
        Friction velocity (m/s)
    theta_star : float
        Temperature scale (K)
    z0 : float
        Roughness length (m)
    
    Returns
    -------
    U, theta : arrays
        Wind speed and potential temperature profiles
    """
    # Dimensionless height
    zeta = z / L_ob  # Negative (stable)
    
    # MOST wind profile (log + stability correction)
    phi_m = 1 + 5 * zeta  # Linear stable form (e.g., Högström 1988)
    U = (u_star / 0.41) * (np.log(z / z0) + 5 * zeta)
    
    # MOST temperature profile
    phi_h = 0.95 + 7.8 * zeta
    theta = THETA_0 + (theta_star / 0.41) * (np.log(z / z0) + 7.8 * zeta)
    
    return U, theta

# Generate synthetic profile
z = np.logspace(-0.5, 2.5, 100)  # 0.3 m to 300 m, 100 points
U, theta = synthetic_stable_layer(z)

# Compute derivatives (numerical)
dU_dz = np.gradient(U, z)
dtheta_dz = np.gradient(theta, z)

# Gradient Richardson number (vectorized)
ri_g = (G / THETA_0) * dtheta_dz / (dU_dz**2 + 1e-10)

print("Synthetic stable layer profile:")
print(f"  Height range: {z.min():.2f} – {z.max():.2f} m")
print(f"  Wind range: {U.min():.2f} – {U.max():.2f} m/s")
print(f"  Temperature range: {theta.min():.2f} – {theta.max():.2f} K")
print(f"  Ri_g range: {ri_g.min():.4f} – {ri_g.max():.4f}")
print(f"  Ri_g(z=10m) ≈ {ri_g[np.argmin(np.abs(z-10))]:.4f}")

## Part 1: Definitions & Physical Interpretation

### Gradient Richardson Number

At a point $z$, the gradient Richardson number measures the ratio of stratification to shear:

$$Ri_g = \frac{N^2}{S^2} = \frac{g}{\theta_0} \frac{\partial\theta/\partial z}{(\partial U/\partial z)^2 + (\partial V/\partial z)^2}$$

**Physical regimes:**
- $Ri_g < 0$: **Unstable** — buoyancy drives convection (mixed layer)
- $Ri_g \approx 0$: **Neutral** — shear and stratification balance
- $0 < Ri_g < Ri_c$ (e.g., $\approx 0.25$): **Stable with turbulence**
- $Ri_g \geq Ri_c$: **Critical layer** — turbulence suppressed, laminar flow

### Bulk Richardson Number

For a layer $z_1 \to z_2$, the bulk Richardson number averages over height:

$$Ri_b = \frac{g}{\theta_0} \frac{(\theta_2 - \theta_1) \cdot (z_2 - z_1)}{(U_2 - U_1)^2 + (V_2 - V_1)^2}$$

**Why two definitions?**
- $Ri_g$: Local diagnostic, requires gradients (needs high-resolution data)
- $Ri_b$: Bulk quantity, used in models (coarse grids, efficiency)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import sys

# Add ABL/src to path for imports
sys.path.insert(0, '/Users/davidengland/Documents/GitHub/ABL/src')

# Import toolkit components (these will work once modules are complete)
try:
    from rct.core import ri_gradient, ri_bulk, bias_ratio
    from rct.core import central_with_curvature, second_derivative
    print("✓ RCT core modules imported successfully")
except ImportError as e:
    print(f"⚠ RCT import partial (expected during development): {e}")

# Physical constants
G = 9.81  # m/s²
VON_KARMAN = 0.41
THETA_0 = 300.0  # Reference temperature (K)

# 1. Introduction to Richardson Numbers & Curvature

**Tutorial:** Gradient vs. Bulk Richardson Numbers, MOST relations, and discretization bias

**Author:** David England (UAH)  
**Date:** December 2025  
**Toolkit:** Richardson Curvature Toolkit (RCT) v0.0.1-alpha

This notebook introduces:
- **Gradient Richardson number** $Ri_g = \frac{g}{\theta_0} \frac{\partial\theta/\partial z}{(\partial U/\partial z)^2}$ — local stability
- **Bulk Richardson number** $Ri_b = \frac{g}{\theta_0} \frac{\Delta\theta \cdot \Delta z}{(\Delta U)^2}$ — layer-averaged stability
- **Bias ratio** $B = Ri_g(z_g) / Ri_b$ — systematic discretization errors
- **Curvature** $\partial^2 Ri_g / \partial\zeta^2$ — how profiles bend with stability

By end: You'll compute both quantities on synthetic stable layer profiles and understand why coarse grids underestimate stability.